# Phase 2: multi-agent system
Run cells in order. 📸 Screenshot the graph, the printed run summaries (agent run logs) and a report file.

In [ ]:
import json, time
from pyspark.sql import functions as F
from src import config
from src.agents import AgentSystem
config.configure_adls(spark)

# Human approval for destructive actions is given here, explicitly, by the person running the notebook.
agents = AgentSystem(spark, approvals={"overwrite": True})

## Agent graph

In [ ]:
from IPython.display import Image
png = agents.app.get_graph().draw_mermaid_png()
config.get_dbutils(spark).fs.put(config.abfss("reports", "agent_graph.mmd"), agents.app.get_graph().draw_mermaid(), overwrite=True)
Image(png)

## Typical paths

In [ ]:
full = agents.run("Run the full pipeline and validate it", mode="full")            # supervisor, etl, quality, analytics

In [ ]:
validate = agents.run("Validate the Bronze and Silver data")                          # supervisor, quality, analytics

In [ ]:
question = agents.run("How many bookings were quarantined in the last run and why?")  # supervisor, explain
print(question["answer"])

## Guardrail: a full reload without human approval is refused

In [ ]:
no_approval = AgentSystem(spark, approvals={})
blocked = no_approval.run("Reload everything from scratch", mode="full")
assert blocked["escalated_to_human"] and blocked["stopped_at"] == "etl"

## Failure scenarios
Each scenario replaces the raw file with a faulty copy, asks the agents for a full run, then restores the original.
The original CSV is moved to `raw/uber_rides_original/` while a scenario runs.

In [ ]:
dbu = config.get_dbutils(spark)
BACKUP = config.abfss("raw", "uber_rides_original/")
original = [f for f in dbu.fs.ls(config.RAW_PATH) if f.name.endswith(".csv")][0]
dbu.fs.cp(original.path, BACKUP + original.name)
clean_raw = spark.read.option("header", "true").option("escape", '"').csv(BACKUP + original.name)   # raw text, nulls kept as 'null'

def write_raw(df):
    for f in dbu.fs.ls(config.RAW_PATH):
        dbu.fs.rm(f.path, True)
    tmp = config.abfss("raw", "_tmp/")
    df.coalesce(1).write.mode("overwrite").option("header", "true").option("escape", '"').csv(tmp)
    part = [f.path for f in dbu.fs.ls(tmp) if f.name.startswith("part-")][0]
    dbu.fs.cp(part, config.RAW_PATH + "rides.csv")
    dbu.fs.rm(tmp, True)

def restore():
    for f in dbu.fs.ls(config.RAW_PATH):
        dbu.fs.rm(f.path, True)
    dbu.fs.cp(BACKUP + original.name, config.RAW_PATH + original.name)

completed = F.col("Booking Status") == "Completed"
SCENARIOS = {
    "healthy":              lambda: restore(),
    "missing_source_file":  lambda: [dbu.fs.rm(f.path, True) for f in dbu.fs.ls(config.RAW_PATH)],
    "schema_change":        lambda: write_raw(clean_raw.withColumnRenamed("Booking Value", "Fare")),
    "duplicate_records":    lambda: write_raw(clean_raw.unionByName(clean_raw.sample(0.05, seed=1))),
    "null_spike":           lambda: write_raw(clean_raw.withColumn("Booking Value", F.when(completed & (F.rand(1) < 0.3), "null").otherwise(F.col("Booking Value")))),
    "out_of_range_values":  lambda: write_raw(clean_raw.withColumn("Booking Value", F.when(completed & (F.rand(1) < 0.02), "-250").otherwise(F.col("Booking Value")))
                                                       .withColumn("Driver Ratings", F.when(completed & (F.rand(2) < 0.02), "6.5").otherwise(F.col("Driver Ratings")))),
    "status_flag_mismatch": lambda: write_raw(clean_raw.withColumn("Cancelled Rides by Driver", F.when(completed & (F.rand(1) < 0.03), "1").otherwise(F.col("Cancelled Rides by Driver")))),
    "row_count_drop":       lambda: write_raw(clean_raw.filter(~F.substring("Date", 6, 2).isin("03", "04", "05"))),
    "job_failure":          lambda: restore(),   # injected below with an invalid job parameter
}
EXPECTED_STOP = {"missing_source_file": "etl", "schema_change": "etl", "job_failure": "etl"}   # others: quality

In [ ]:
results = []
for name, inject in SCENARIOS.items():
    inject()
    try:
        mode = "not-a-mode" if name == "job_failure" else "full"
        r = agents.run(f"Run the full pipeline ({name} test)", mode=mode)
    finally:
        restore()
    expected_stop = EXPECTED_STOP.get(name, "quality")
    explained = all(f.get("explanation") and f.get("recommended_action") for f in r["findings"])
    results.append({"scenario": name, "status": r["status"], "stopped_at": r["stopped_at"],
                    "detected": r["status"] == "FAIL",
                    "recovered": r["status"] == "FAIL" and r["stopped_at"] == expected_stop and explained,
                    "tokens": r["tokens_used"], "seconds": r["seconds"]})
display(spark.createDataFrame([{k: str(v) for k, v in x.items()} for x in results]))

## Evaluation metrics
Report quality is reviewed manually from the findings printed above.
Set `MANUAL_MINUTES` to your own estimate of checking one load by hand.

In [ ]:
MANUAL_MINUTES = 45
faults = [x for x in results if x["scenario"] != "healthy"]
healthy = [x for x in results if x["scenario"] == "healthy"]
metrics = {
    "detection_rate": f"{sum(x['detected'] for x in faults)}/{len(faults)}",
    "false_alarm_rate": f"{sum(x['detected'] for x in healthy)}/{len(healthy)}",
    "recovery_rate": f"{sum(x['recovered'] for x in faults)}/{len(faults)}",
    "avg_agent_minutes_per_run": round(sum(x["seconds"] for x in results) / len(results) / 60, 1),
    "manual_minutes_per_run": MANUAL_MINUTES,
    "avg_tokens_per_run": round(sum(x["tokens"] for x in results) / len(results)),
}
print(json.dumps(metrics, indent=2))

In [ ]:
# leave the platform healthy and remove the backup
restore()
dbu.fs.rm(BACKUP, True)
agents.run("Run the full pipeline and validate it", mode="full")